# Baseline: QLoRA Text-to-SVG Generation

**NYU Deep Learning — Spring 2026 Midterm Kaggle Competition**

This notebook implements a complete baseline pipeline in **two phases**:

**Phase A — Training** (requires internet & GPU):

1. Environment Setup — dependencies, seeds, configuration
2. SVG Utilities — competition-compliant validation, post-processing, fallback
3. Data Pipeline — multi-source loading, normalization, quality filtering
4. Model Setup — Qwen 2B + QLoRA via Unsloth
5. Training — SFT with chat-formatted (prompt, SVG) pairs

**Phase B — Inference & Submission** (can run offline on Kaggle):

6. Inference & Submission — generate, validate, export `submission.csv`

> **For Kaggle Code Submission:** split at the phase boundary. Upload the trained adapter as a Kaggle dataset, then create a separate offline notebook that only runs Phase B.

In [1]:
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ET.register_namespace('', 'http://www.w3.org/2000/svg')
ET.register_namespace('xlink', 'http://www.w3.org/1999/xlink')

print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
CONFIG = {
    # Model
    'model_name': 'unsloth/Qwen2.5-3B-Instruct',
    'max_seq_length': 2048,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0,
    'target_modules': [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],

    # Training
    'learning_rate': 2e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 32,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'eval_steps': 100,
    'save_steps': 200,

    # Data
    'max_train_samples_per_source': 12000,
    'max_svg_train_chars': 2500,
    'eval_size': 0.02,

    # SVG constraints (competition rules)
    'max_svg_length': 16000,
    'max_path_count': 256,
    'canvas_size': 256,

    # Inference
    'inference_batch_size': 4,
    'max_new_tokens': 768,
    'temperature': 0.3,
    'top_p': 0.8,
    'repetition_penalty': 1.05,

    
    # Paths — Colab + Google Drive
    #'train_data_path': '/content/drive/MyDrive/midterm/train.csv',
    #'output_dir': '/content/drive/MyDrive/midterm/qwen25_3b_svg_lora',
    #'test_prompts_path': '/content/drive/MyDrive/midterm/test.csv',
    #'submission_path': '/content/drive/MyDrive/midterm/submission.csv',
    
    
    # Paths — Kaggle
    'train_data_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv',
    'output_dir': '/kaggle/working/qwen25_3b_svg_lora',
    'test_prompts_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv',
    'submission_path': '/kaggle/working/submission.csv',
}

CONFIG

{'model_name': 'unsloth/Qwen2.5-3B-Instruct',
 'max_seq_length': 2048,
 'lora_r': 16,
 'lora_alpha': 16,
 'lora_dropout': 0,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 32,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 12000,
 'max_svg_train_chars': 2500,
 'eval_size': 0.02,
 'max_svg_length': 16000,
 'max_path_count': 256,
 'canvas_size': 256,
 'inference_batch_size': 4,
 'max_new_tokens': 768,
 'temperature': 0.3,
 'top_p': 0.8,
 'repetition_penalty': 1.05,
 'train_data_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv',
 'output_dir': '/kaggle/working/qwen25_3b_svg_lora',
 'test_prompts_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv',
 'submission_path': '/ka

---
## 2. SVG Utilities

Competition-compliant validation, post-processing, and fallback.

Key constraints: 256×256 canvas, ≤16 000 chars, ≤256 paths, whitelist-only tags, no scripts/events/animation.

In [4]:
ALLOWED_TAGS = frozenset({
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter',
})


def _strip_ns(tag):
    return tag.split('}', 1)[-1] if '}' in tag else tag


def _count_paths(root):
    return sum(1 for e in root.iter() if _strip_ns(e.tag) == 'path')


def _collect_bad_tags(root):
    return {_strip_ns(e.tag) for e in root.iter() if _strip_ns(e.tag) not in ALLOWED_TAGS}


def _has_event_handlers(root):
    for e in root.iter():
        for attr in e.attrib:
            local = attr.split('}')[-1] if '}' in attr else attr
            if local.lower().startswith('on'):
                return True
    return False


def validate_svg(svg_text, max_length=16000, max_paths=256):
    """
    Competition-compliant SVG validation.
    Returns (is_valid, reason).
    """
    if not svg_text or not isinstance(svg_text, str):
        return False, 'empty'
    if len(svg_text) > max_length:
        return False, f'too_long ({len(svg_text)})'
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError as e:
        return False, f'parse_error: {e}'
    if _strip_ns(root.tag) != 'svg':
        return False, f'root_not_svg ({root.tag})'
    bad = _collect_bad_tags(root)
    if bad:
        return False, f'bad_tags: {bad}'
    if _has_event_handlers(root):
        return False, 'event_handlers'
    n = _count_paths(root)
    if n > max_paths:
        return False, f'too_many_paths ({n})'
    return True, 'ok'


def sanitize_svg(svg_text):
    """
    Post-process SVG: fix root attributes, strip disallowed elements/attributes.
    Returns cleaned SVG string or empty string on failure.
    """
    if not svg_text:
        return ''
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return ''
    if _strip_ns(root.tag) != 'svg':
        return ''
    if '{' not in root.tag:
        root.tag = '{http://www.w3.org/2000/svg}svg'
    root.attrib.pop('xmlns', None)
    root.set('width', '256')
    root.set('height', '256')
    root.set('viewBox', '0 0 256 256')
    _remove_bad_elements(root)
    _remove_event_attrs(root)
    return ET.tostring(root, encoding='unicode')


def _remove_bad_elements(elem):
    to_remove = [c for c in elem if _strip_ns(c.tag) not in ALLOWED_TAGS]
    for c in to_remove:
        elem.remove(c)
    for c in elem:
        _remove_bad_elements(c)


def _remove_event_attrs(root):
    for e in root.iter():
        bad = [a for a in e.attrib
               if (a.split('}')[-1] if '}' in a else a).lower().startswith('on')]
        for a in bad:
            del e.attrib[a]


def fallback_svg(prompt=''):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" '
        'viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="gray"/>'
        '</svg>'
    )


_ok, _reason = validate_svg(fallback_svg())
print(f'Fallback SVG valid: {_ok} ({_reason}), length: {len(fallback_svg())} chars')

Fallback SVG valid: True (ok), length: 196 chars


---
## 3. Data Pipeline

Load competition `train.csv`, normalize SVGs to 256×256, and apply quality filters.

In [5]:
_PROMPT_PREFIX_RE = re.compile(
    r'^(?:Generate|Create|Write|Make|Draw|Produce)\s+'
    r'(?:(?:the\s+|an?\s+)?(?:SVG|svg)\s+(?:code|image|graphic)\s+)?'
    r'(?:for\s+)?(?:an?\s+image\s+(?:that|which)\s+looks?\s+like[:\s]*)?',
    re.IGNORECASE,
)
_PROMPT_SUFFIX_RE = re.compile(
    r"[\.\s]*(?:Don'?t|do\s+not)\s+use\s+markdown.*$",
    re.IGNORECASE,
)


def _clean_prompt(prompt):
    """Strip instruction wrappers, keeping only the visual description."""
    prompt = _PROMPT_PREFIX_RE.sub('', prompt)
    prompt = _PROMPT_SUFFIX_RE.sub('', prompt)
    return prompt.strip()


def _quality_filter(example):
    """Keep only samples with valid, reasonably-sized SVGs."""
    svg = example.get('svg', '')
    prompt = example.get('prompt', '')
    if not svg or not prompt:
        return False
    if len(svg) > CONFIG['max_svg_train_chars']:
        return False
    ok, _ = validate_svg(svg)
    return ok

In [6]:
# ── Skip-training guard ──
import os as _os
_KAGGLE_INPUT_LORA = '/kaggle/input/models/sxunyu/qwen25-3b-svg-lora/transformers/default/1'
_adapter_working = _os.path.join(CONFIG["output_dir"], "adapter_config.json")
_adapter_kaggle  = _os.path.join(_KAGGLE_INPUT_LORA,   "adapter_config.json")
if _os.path.exists(_adapter_kaggle):
    SKIP_TRAINING = True
    print(f"[SKIP] Kaggle input adapter found: {_KAGGLE_INPUT_LORA}")
    print("       Cells 11, 13, 14 will be skipped. Jump to Cell 17 for inference.")
elif _os.path.exists(_adapter_working):
    SKIP_TRAINING = True
    print(f"[SKIP] Working-dir adapter found: {CONFIG['output_dir']}")
    print("       Cells 11, 13, 14 will be skipped. Jump to Cell 17 for inference.")
else:
    SKIP_TRAINING = False
    print("[TRAIN] No adapter found — will train from scratch.")

# ── Load competition train.csv as primary data source ──
comp_df = pd.read_csv(CONFIG['train_data_path'])
print(f'Competition train.csv: {len(comp_df)} rows, columns={list(comp_df.columns)}')

comp_ds = Dataset.from_pandas(comp_df[['prompt', 'svg']])


def _normalize_comp(example):
    prompt = str(example.get('prompt', '')).strip()
    prompt = _clean_prompt(prompt)
    svg = sanitize_svg(str(example.get('svg', '')))
    return {'prompt': prompt, 'svg': svg}


comp_ds = comp_ds.map(_normalize_comp, remove_columns=comp_ds.column_names,
                      desc='normalizing train.csv')
before = len(comp_ds)
comp_ds = comp_ds.filter(_quality_filter, desc='filtering train.csv')
print(f'  Usable: {len(comp_ds)} rows (dropped {before - len(comp_ds)})')

max_samples = CONFIG['max_train_samples_per_source']
if max_samples and len(comp_ds) > max_samples:
    comp_ds = comp_ds.shuffle(seed=SEED).select(range(max_samples))
    print(f'  Subsampled to {len(comp_ds)} rows')

train_raw = comp_ds.shuffle(seed=SEED)
splits = train_raw.train_test_split(test_size=CONFIG['eval_size'], seed=SEED)
train_ds = splits['train']
eval_ds = splits['test']

print(f'\nTrain: {len(train_ds)} | Eval: {len(eval_ds)}')
if len(train_ds) > 0:
    print(f'Sample prompt: {train_ds[0]["prompt"][:120]}')
    print(f'Sample SVG (first 200): {train_ds[0]["svg"][:200]}')
else:
    raise RuntimeError(
        'Training set is empty after filtering. '
        'Check max_svg_train_chars / validate_svg settings.'
    )

[SKIP] Kaggle input adapter found: /kaggle/input/models/sxunyu/qwen25-3b-svg-lora/transformers/default/1
       Cells 11, 13, 14 will be skipped. Jump to Cell 17 for inference.
Competition train.csv: 50000 rows, columns=['id', 'prompt', 'svg']


normalizing train.csv:   0%|          | 0/50000 [00:00<?, ? examples/s]

filtering train.csv:   0%|          | 0/50000 [00:00<?, ? examples/s]

  Usable: 29332 rows (dropped 20668)
  Subsampled to 12000 rows

Train: 11760 | Eval: 240
Sample prompt: Black outline of an airplane in flight above a shadow.
Sample SVG (first 200): <svg xmlns="http://www.w3.org/2000/svg" height="256" preserveAspectRatio="xMidYMid meet" viewBox="0 0 256 256" width="256"><path d="m61.94 5.97c-.38-1.86-2.49-3.11-5.95-3.5-2.29-.25-4.34-.47-6.16-.47-


In [7]:
SYSTEM_PROMPT = (
    'You generate compact, valid SVG code from text descriptions. '
    'Output only a single <svg> element with xmlns="http://www.w3.org/2000/svg", '
    'width="256", height="256", viewBox="0 0 256 256". '
    'No explanation, no markdown — only SVG code.'
)


def format_sft(example):
    text = (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{example["prompt"]}<|im_end|>\n'
        '<|im_start|>assistant\n'
        f'{example["svg"]}<|im_end|>'
    )
    return {'text': text}


train_text = train_ds.map(format_sft, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft, remove_columns=eval_ds.column_names)

print('Formatted sample (first 500 chars):')
print(train_text[0]['text'][:500])

Map:   0%|          | 0/11760 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Formatted sample (first 500 chars):
<|im_start|>system
You generate compact, valid SVG code from text descriptions. Output only a single <svg> element with xmlns="http://www.w3.org/2000/svg", width="256", height="256", viewBox="0 0 256 256". No explanation, no markdown — only SVG code.<|im_end|>
<|im_start|>user
Black outline of an airplane in flight above a shadow.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" height="256" preserveAspectRatio="xMidYMid meet" viewBox="0 0 256 256" width="256"><path d="m61


---
## 4. Model Setup — Qwen 2B + QLoRA (Unsloth)

In [8]:
if not SKIP_TRAINING:
    from unsloth import FastLanguageModel
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CONFIG['model_name'],
        max_seq_length=CONFIG['max_seq_length'],
        dtype=None,
        load_in_4bit=True,
    )
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=CONFIG['lora_r'],
        lora_alpha=CONFIG['lora_alpha'],
        lora_dropout=CONFIG['lora_dropout'],
        bias='none',
        target_modules=CONFIG['target_modules'],
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )
    
    model.print_trainable_parameters()

else:
    print("[SKIP] Model setup skipped (adapter already exists).")


[SKIP] Model setup skipped (adapter already exists).


---
## 5. Training

In [9]:
if not SKIP_TRAINING:
    from transformers import TrainingArguments, Trainer as _HFTrainer
    from trl import SFTTrainer
    
    # ============ Fix: Unsloth / transformers >=4.46 compatibility ============
    # transformers passes num_items_in_batch as int; Unsloth calls .mean() on it.
    # Strategy A — class-level patch on Trainer.training_step (survives compiled cache)
    _orig_trainer_step = _HFTrainer.training_step
    
    def _safe_training_step(self, model, inputs, num_items_in_batch=None):
        if isinstance(num_items_in_batch, (int, float)):
            num_items_in_batch = torch.tensor(float(num_items_in_batch))
        return _orig_trainer_step(self, model, inputs, num_items_in_batch)
    
    _HFTrainer.training_step = _safe_training_step
    print("[patch] Trainer.training_step wrapped for int→tensor fix")
    # ==========================================================================
    
    training_args = TrainingArguments(
        output_dir=CONFIG['output_dir'],
        num_train_epochs=CONFIG['num_train_epochs'],
        per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        learning_rate=CONFIG['learning_rate'],
        warmup_ratio=CONFIG['warmup_ratio'],
        weight_decay=CONFIG['weight_decay'],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=CONFIG['logging_steps'],
        eval_strategy='steps',
        eval_steps=CONFIG['eval_steps'],
        save_steps=CONFIG['save_steps'],
        save_total_limit=2,
        report_to='none',
        optim='paged_adamw_8bit',
        lr_scheduler_type='cosine',
        seed=SEED,
    )
    
    _resp_tpl = '<|im_start|>assistant\n'
    print(f'[info] response_template = {_resp_tpl!r}')
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_text,
        eval_dataset=eval_text,
        dataset_text_field='text',
        max_seq_length=CONFIG['max_seq_length'],
        packing=False,
        args=training_args,
        response_template=_resp_tpl,
    )
    
    # Strategy B — disable num_items_in_batch from being passed at all
    trainer.model_accepts_loss_kwargs = False
    print("[patch] model_accepts_loss_kwargs = False  →  num_items_in_batch disabled")
    
    train_result = trainer.train()
    print(train_result)

else:
    print("[SKIP] Training skipped (adapter already exists).")


[SKIP] Training skipped (adapter already exists).


In [10]:
if not SKIP_TRAINING:
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    trainer.save_model(CONFIG['output_dir'])
    tokenizer.save_pretrained(CONFIG['output_dir'])
    print(f'Adapter + tokenizer saved to {CONFIG["output_dir"]}')

else:
    print("[SKIP] Save skipped (adapter already exists).")


[SKIP] Save skipped (adapter already exists).


In [11]:
if not SKIP_TRAINING:
    import gc
    
    # Free training artifacts — optimizer states, gradients, cached activations
    del trainer, training_args
    gc.collect()
    torch.cuda.empty_cache()
    
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory after cleanup: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

else:
    import gc; gc.collect()
    print("[SKIP] Cleanup skipped (no training artifacts to free).")


[SKIP] Cleanup skipped (no training artifacts to free).


---
## Phase B: Inference & Submission

> Everything below this line can run **offline** on Kaggle.
> For the actual Kaggle code submission, create a separate notebook that loads the pre-trained adapter from a Kaggle dataset and only executes the cells below.

In [ ]:
import gc, importlib, time, re
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from peft import PeftModel

# ── Path configuration: automatically select the LoRA adapter path ──
_KAGGLE_INPUT_LORA = '/kaggle/input/models/sxunyu/qwen25-3b-svg-lora/transformers/default/1'
if os.path.exists(os.path.join(_KAGGLE_INPUT_LORA, "adapter_config.json")):
    _LORA_DIR = _KAGGLE_INPUT_LORA
    print(f"[PATH] Using Kaggle input LoRA: {_LORA_DIR}")
else:
    _LORA_DIR = CONFIG['output_dir']
    print(f"[PATH] Using working-dir LoRA: {_LORA_DIR}")

# ════════════════════════════════════════════════════════════
# Phase 1: Environment cleanup — unconditionally release training-state objects
# ════════════════════════════════════════════════════════════

# 1a. Clear Unsloth's global patches on qwen2 modules
import transformers.models.qwen2.modeling_qwen2 as _qwen2
importlib.reload(_qwen2)
print('[P1] Reloaded qwen2 module — Unsloth forward patches removed')

# 1b. Delete all training-state model objects 
try:
    del model
except NameError:
    pass
try:
    del base_model
except NameError:
    pass
try:
    del trainer
except NameError:
    pass
try:
    del peft_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
_free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f'[P1] GPU cleared. Free VRAM: {_free_gb:.1f} GB')

# ════════════════════════════════════════════════════════════
# Phase 2: Load a clean model + merge LoRA
# ════════════════════════════════════════════════════════════

# 2a. Load the base model from HuggingFace, explicitly specifying attn_implementation
#     This is more reliable than setting config._attn_implementation post-hoc
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    device_map='auto',
    dtype=torch.float16,
    attn_implementation='sdpa',
)
print(f'[P2] Base model loaded: {type(base_model).__name__}')

# 2b. Fully replace __class__ with the clean HF class (belt-and-suspenders)
#     Even if importlib.reload has restored the module-level definitions, from_pretrained may
#     still reference the old class patched by Unsloth through a cached AUTO_MODEL_MAPPING
base_model.__class__ = _qwen2.Qwen2ForCausalLM
base_model.model.__class__ = _qwen2.Qwen2Model
_n_layers = len(base_model.model.layers)
for _layer in base_model.model.layers:
    _layer.__class__ = _qwen2.Qwen2DecoderLayer
    _layer.self_attn.__class__ = _qwen2.Qwen2Attention
    _layer.mlp.__class__ = _qwen2.Qwen2MLP
    _layer.input_layernorm.__class__ = _qwen2.Qwen2RMSNorm
    _layer.post_attention_layernorm.__class__ = _qwen2.Qwen2RMSNorm
base_model.model.norm.__class__ = _qwen2.Qwen2RMSNorm
base_model.config._attn_implementation = 'sdpa'
print(f'[P2] Class-swapped {_n_layers} layers → clean HF + sdpa')

# 2c. Load the LoRA adapter and merge it into the base weights
#     merge_and_unload() removes the 252-layer per-token Python dispatch overhead
#     from the PeftModel wrapper, which was one of the main causes of the previous
#     slow speed of 9.5 tok/s
_peft = PeftModel.from_pretrained(base_model, _LORA_DIR)
model = _peft.merge_and_unload()
del _peft, base_model
gc.collect()
# In newer versions of PEFT, the peft_config attribute may remain on the model
# after merge_and_unload(). This does not affect inference, but can cause
# hasattr checks to give false positives. Remove it explicitly.
if hasattr(model, 'peft_config'):
    del model.peft_config
print(f'[P2] LoRA merged into base weights — plain {type(model).__name__}')

# 2d. Remove all remaining Unsloth forward hooks
_hooks_cleared = 0
for _m in model.modules():
    _hooks_cleared += len(_m._forward_hooks) + len(_m._forward_pre_hooks)
    _m._forward_hooks.clear()
    _m._forward_pre_hooks.clear()
print(f'[P2] Cleared {_hooks_cleared} residual hooks')

model.eval()

# ════════════════════════════════════════════════════════════
# Phase 3: Tokenizer + Generation Config (resolve P3)
# ════════════════════════════════════════════════════════════

tokenizer = AutoTokenizer.from_pretrained(_LORA_DIR)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3a. Build a list of EOS token IDs
_eos_ids = []
if hasattr(model.config, 'eos_token_id'):
    v = model.config.eos_token_id
    _eos_ids = list(v) if isinstance(v, list) else [v]
_im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
if _im_end_id and _im_end_id not in _eos_ids:
    _eos_ids.append(_im_end_id)
EOS_TOKEN_IDS = _eos_ids

# 3b. Reset generation_config to resolve the max_length=32768 conflict
#     Previously, setting model.generation_config.max_length = None did not work
#     because Qwen2.5's generation_config.json enforces max_length=32768.
#     Correct approach: replace it with a new GenerationConfig, keeping only essential parameters.
model.generation_config = GenerationConfig(
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=EOS_TOKEN_IDS,
    max_new_tokens=512,
)
print(f'[P3] generation_config reset (no max_length conflict)')
print(f'[P3] EOS token IDs: {EOS_TOKEN_IDS}')

# ════════════════════════════════════════════════════════════
# Phase 4: Assertion checks — verify all preconditions before spending GPU time
# ════════════════════════════════════════════════════════════

_impl = getattr(model.config, '_attn_implementation', 'unknown')
_dtype = next(model.parameters()).dtype
_on_gpu = all(p.device.type == 'cuda' for p in model.parameters())
_cls = type(model).__name__
_alloc = torch.cuda.memory_allocated() / 1e9

print(f'\n{"=" * 55}')
print(f'  INFERENCE MODEL DIAGNOSTICS')
print(f'{"=" * 55}')
print(f'  Model class:       {_cls}')
print(f'  attn_impl:         {_impl}')
print(f'  dtype:             {_dtype}')
print(f'  All on GPU:        {_on_gpu}')
print(f'  GPU mem allocated:  {_alloc:.2f} GB')
print(f'  Is PeftModel:      {hasattr(model, "peft_config")}')
print(f'  EOS IDs:           {EOS_TOKEN_IDS}')
print(f'  gen_config.max_length: {getattr(model.generation_config, "max_length", "NOT SET")}')
print(f'{"=" * 55}')

assert _cls == 'Qwen2ForCausalLM', f'Expected Qwen2ForCausalLM, got {_cls}'
assert _impl == 'sdpa', f'Expected sdpa, got {_impl}'
assert _dtype == torch.float16, f'Expected fp16, got {_dtype}'
assert _on_gpu, 'Some params on CPU!'
assert not hasattr(model, 'peft_config'), 'LoRA not merged!'

# 4b. Forward pass speed benchmark
_test_ids = tokenizer('hello', return_tensors='pt').input_ids.to(model.device)
torch.cuda.synchronize()
_t0 = time.time()
with torch.no_grad():
    for _ in range(20):
        model(_test_ids)
torch.cuda.synchronize()
_fwd_ms = (time.time() - _t0) / 20 * 1000
print(f'\n  Forward latency:   {_fwd_ms:.1f} ms/step')
if _fwd_ms > 50:
    print(f'  *** WARNING: {_fwd_ms:.0f}ms too high. Expected <20ms A100, <50ms T4.')
else:
    print(f'  Forward speed OK.')
print()

# ════════════════════════════════════════════════════════════
# Phase 5: Inference helpers
# ════════════════════════════════════════════════════════════

SVG_EXTRACT_RE = re.compile(r'<svg.*?</svg>', flags=re.IGNORECASE | re.DOTALL)

GENERATE_KWARGS = dict(
    max_new_tokens=min(CONFIG['max_new_tokens'], 512),
    do_sample=True,
    temperature=CONFIG['temperature'],
    top_p=CONFIG['top_p'],
    repetition_penalty=max(CONFIG.get('repetition_penalty', 1.05), 1.15),
    eos_token_id=EOS_TOKEN_IDS,
    pad_token_id=tokenizer.pad_token_id,
)
print(f'[P5] max_new_tokens={GENERATE_KWARGS["max_new_tokens"]}, '
      f'repetition_penalty={GENERATE_KWARGS["repetition_penalty"]}')


def extract_svg(text):
    """Extract SVG. If the model output is truncated (missing </svg>), attempt to force-close it at the last complete element."""
    m = SVG_EXTRACT_RE.search(text)
    if m:
        return m.group(0).strip()
    # Fallback: the model hit max tokens and generated <svg>... but didn’t finish outputting </svg>
    idx = text.lower().find('<svg')
    if idx < 0:
        return ''
    partial = text[idx:]
    last_selfclose = partial.rfind('/>')
    last_closetag = partial.rfind('</')
    if last_closetag > last_selfclose:
        end_gt = partial.find('>', last_closetag)
        if end_gt > 0:
            cut = end_gt + 1
        else:
            cut = last_selfclose + 2 if last_selfclose > 0 else -1
    elif last_selfclose > 0:
        cut = last_selfclose + 2
    else:
        # Last resort: if the model gets stuck inside a path d="..." segment,
        # force-close the path and the svg.
        # Find the start of the last <path or other <... tag and truncate the d attribute.
        last_open_tag = max(partial.rfind('<path'), partial.rfind('<circle'),
                            partial.rfind('<rect'), partial.rfind('<ellipse'),
                            partial.rfind('<line'), partial.rfind('<polyline'),
                            partial.rfind('<polygon'))
        if last_open_tag > 0:
            recovered = partial[:last_open_tag].rstrip() + '\n</svg>'
            return recovered.strip()
        return ''
    recovered = partial[:cut] + '\n</svg>'
    return recovered.strip()


def _postprocess(decoded, prompt, debug=False):
    """Return (svg, status). If debug=True, print the intermediate result at each step."""
    raw = extract_svg(decoded)
    if not raw:
        if debug:
            print(f'    [FAIL] extract_svg returned empty')
            print(f'    [FAIL] decoded text (first 500 chars):')
            print(f'           {decoded[:500]}')
        return fallback_svg(prompt), 'extract_fail'

    cleaned = sanitize_svg(raw)
    if not cleaned:
        if debug:
            print(f'    [FAIL] sanitize_svg returned empty')
            print(f'    [FAIL] raw SVG (first 300 chars): {raw[:300]}')
        return fallback_svg(prompt), 'sanitize_fail'

    ok, reason = validate_svg(cleaned)
    if not ok:
        if debug:
            print(f'    [FAIL] validate_svg: {reason}')
            print(f'    [FAIL] cleaned SVG (first 300 chars): {cleaned[:300]}')
        return fallback_svg(prompt), f'validate_fail:{reason}'

    if debug:
        print(f'    [OK] valid SVG, {len(cleaned)} chars')
    return cleaned, 'ok'


def generate_svg(prompt, debug=False):
    """Run inference on a single input. If debug=True, print generation details."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    chat = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(chat, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[-1]

    with torch.no_grad():
        out = model.generate(**inputs, **GENERATE_KWARGS)

    gen_ids = out[0][input_len:]
    n_gen = len(gen_ids)
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

    if debug:
        _actual_max = GENERATE_KWARGS['max_new_tokens']
        hit_eos = n_gen < _actual_max
        print(f'  [gen] {n_gen} tokens ({"stopped at EOS" if hit_eos else f"HIT MAX={_actual_max}"})')
        print(f'  [gen] decoded len={len(decoded)}, first 300 chars:')
        print(f'        {decoded[:300]}')

    return _postprocess(decoded, prompt, debug=debug)


def generate_svg_batch(prompts):
    """Run inference one sample at a time to avoid quality degradation caused by left padding."""
    all_results = []
    for p in prompts:
        svg, status = generate_svg(p, debug=False)
        all_results.append((svg, status))
    return all_results


print('Inference setup complete. Ready for smoke test.\n')


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


[PATH] Using Kaggle input LoRA: /kaggle/input/models/sxunyu/qwen25-3b-svg-lora/transformers/default/1
[P1] Reloaded qwen2 module — Unsloth forward patches removed
[P1] GPU cleared. Free VRAM: 15.5 GB


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

[P2] Base model loaded: Qwen2ForCausalLM
[P2] Class-swapped 36 layers → clean HF + sdpa
[P2] LoRA merged into base weights — plain Qwen2ForCausalLM
[P2] Cleared 0 residual hooks
[P3] generation_config reset (no max_length conflict)
[P3] EOS token IDs: [151645]

  INFERENCE MODEL DIAGNOSTICS
  Model class:       Qwen2ForCausalLM
  attn_impl:         sdpa
  dtype:             torch.float16
  All on GPU:        True
  GPU mem allocated:  3.10 GB
  Is PeftModel:      False
  EOS IDs:           [151645]
  gen_config.max_length: None

  Forward latency:   90.2 ms/step
  *** WARNING: 90ms too high. Expected <20ms A100, <50ms T4.

[P5] max_new_tokens=512, repetition_penalty=1.15
Inference setup complete. Ready for smoke test.



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Smoke Test with Full Diagnostics
# Intended to replace the Smoke Test cell in baseline-qwen2-5.ipynb
#
# Compared with the old version, this one adds:
# 1. Full generation diagnostics for each test prompt
#    (token count, whether EOS was reached, raw text)
# 2. Failure reasons printed at each step of extract/sanitize/validate
# 3. Fix for the old bug where model.base_model.model no longer exists
#    after merge_and_unload()
# 4. Uses GENERATE_KWARGS (without max_length) to avoid warning spam
# ═══════════════════════════════════════════════════════════════════
print('=' * 60)
print('SMOKE TEST: Pre-submission Diagnostics')
print('=' * 60)

# ── 1. Environment Checks ──
_impl = getattr(model.config, '_attn_implementation', 'unknown')
_cls = type(model).__name__
_dtype = next(model.parameters()).dtype
_dev = next(model.parameters()).device

print(f'\n[ENV] Model class:       {_cls}')
print(f'[ENV] attn_impl:         {_impl}')
print(f'[ENV] dtype:             {_dtype}')
print(f'[ENV] Device:            {_dev}')
print(f'[ENV] EOS IDs:           {EOS_TOKEN_IDS}')
print(f'[ENV] Batch size:        {CONFIG.get("inference_batch_size", 4)}')
print(f'[ENV] max_new_tokens:    {CONFIG["max_new_tokens"]}')
print(f'[ENV] gen_config.max_length: {getattr(model.generation_config, "max_length", "NOT SET")}')

_problems = []
if _impl != 'sdpa':
    _problems.append(f'attn_implementation={_impl}, expected sdpa')
if _dtype != torch.float16:
    _problems.append(f'dtype={_dtype}, expected fp16')
if hasattr(model, 'peft_config'):
    _problems.append('LoRA not merged (PeftModel still active)')
if getattr(model.generation_config, 'max_length', None) is not None:
    _problems.append(f'generation_config.max_length={model.generation_config.max_length} (will cause warnings)')

if _problems:
    print(f'\n*** {len(_problems)} PROBLEM(S) DETECTED ***')
    for p in _problems:
        print(f'  - {p}')
else:
    print('\n[OK] All environment checks passed.')

# ── 2. Detailed single-sample inference test (debug=True) ──
test_prompts = [
    'a simple red circle on white background',
    'a green tree with brown trunk',
    'a blue five-pointed star icon',
]

print(f'\n{"─" * 60}')
print(f'Running {len(test_prompts)} test prompts with FULL DIAGNOSTICS...\n')

single_results = []
t0 = time.time()

for idx, p in enumerate(test_prompts):
    print(f'--- Prompt {idx+1}/{len(test_prompts)}: "{p}" ---')
    t1 = time.time()
    svg, status = generate_svg(p, debug=True)
    elapsed = time.time() - t1
    single_results.append((p, svg, status, elapsed))
    print(f'  Result: [{status}] {elapsed:.1f}s, SVG len={len(svg)}\n')

single_total = time.time() - t0
print(f'Single-mode total: {single_total:.1f}s for {len(test_prompts)} prompts')

# ── 3. Check EOS behavior (based on the debug=True output above) ──
# If the runtime for all test prompts is close to max_new_tokens / speed,
# it indicates that EOS is never generated
_avg_time = single_total / max(len(test_prompts), 1)
_expected_max_time = CONFIG['max_new_tokens'] / 30  # 30 tok/s baseline
if _avg_time > _expected_max_time * 0.8:
    print(f'\n*** WARNING: Avg {_avg_time:.1f}s/prompt suggests model always generates max_new_tokens. ***')
    print('    EOS/<|im_end|> may never be generated (training quality issue).')
else:
    print(f'\n[OK] Avg {_avg_time:.1f}s/prompt — model appears to stop before max_new_tokens.')

# ── 4. Batch inference test ──
bs = CONFIG.get('inference_batch_size', 4)
print(f'\n{"─" * 60}')
print(f'Running batch test (batch_size={bs})...\n')

t0 = time.time()
batch_results = generate_svg_batch(test_prompts)
batch_elapsed = time.time() - t0

for p, (svg, status) in zip(test_prompts, batch_results):
    print(f'  [{status:>20}] len={len(svg):>5}  prompt="{p}"')

print(f'\n  Batch total: {batch_elapsed:.1f}s for {len(test_prompts)} prompts')

# ── 5. Submission time estimation ──
per_prompt = batch_elapsed / max(len(test_prompts), 1)
total_prompts = 1000
est_min = per_prompt * total_prompts / 60

print(f'\n{"═" * 60}')
print(f'SUBMISSION ESTIMATE')
print(f'  Speed:      {per_prompt:.2f} s/prompt')
print(f'  Prompts:    {total_prompts}')
print(f'  ETA:        {est_min:.0f} min ({est_min/60:.1f} hours)')
print(f'{"═" * 60}')

# ── 6. Quality summary ──
valid = sum(1 for _, s in batch_results if s == 'ok')
print(f'\n[QUALITY] Valid: {valid}/{len(batch_results)}')
for p, (svg, status) in zip(test_prompts, batch_results):
    if status == 'ok':
        print(f'  OK   "{p}" → {len(svg)} chars')
    else:
        print(f'  FAIL "{p}" → {status}')

# ── 7. Final suggestion ──
print(f'\n{"─" * 60}')
if valid == 0 and len(batch_results) > 0:
    print('*** ALL test SVGs failed. Check the diagnostic output above. ***')
    print('    Most common causes:')
    print('    1. Model generates text but not SVGs (training quality issue)')
    print('    2. Model generates SVGs but they fail XML parsing (sanitize_svg)')
    print('    3. Training data format mismatch (check format_chat template)')
    print('    Run the detailed diagnostics above to determine which.')
if est_min > 120:
    print(f'\n*** Estimated {est_min:.0f} min is too slow for Kaggle (limit ~540 min). ***')
elif est_min > 60:
    print(f'\nEstimated {est_min:.0f} min — within limits but could be faster.')
else:
    print(f'\nSpeed looks good for submission.')
print()


SMOKE TEST: Pre-submission Diagnostics

[ENV] Model class:       Qwen2ForCausalLM
[ENV] attn_impl:         sdpa
[ENV] dtype:             torch.float16
[ENV] Device:            cuda:0
[ENV] EOS IDs:           [151645]
[ENV] Batch size:        4
[ENV] max_new_tokens:    768
[ENV] gen_config.max_length: None

[OK] All environment checks passed.

────────────────────────────────────────────────────────────
Running 3 test prompts with FULL DIAGNOSTICS...

--- Prompt 1/3: "a simple red circle on white background" ---
  [gen] 186 tokens (stopped at EOS)
  [gen] decoded len=251, first 300 chars:
        <svg xmlns="http://www.w3.org/2000/svg" fill="#ff0000" height="256" viewBox="0 0 256 256" width="256"><path d="m179.84 105.9c-1.07-.39-2.11-.77-3.12-1.14l-12.98-12.98h-12.98v-12.98c0-1.07.39-2.11.77-3.12s1.14-.77 3.12-1.14l12.98-12.98h12.98z" /></svg>
    [OK] valid SVG, 251 chars
  Result: [ok] 12.5s, SVG len=251

--- Prompt 2/3: "a green tree with brown trunk" ---
  [gen] 512 tokens (HIT MAX=

---
## 7. Generate Submission

In [14]:
test_df = pd.read_csv(CONFIG['test_prompts_path'])
print(f'Test prompts: {len(test_df)}')

# Suppress the noisy "max_new_tokens vs max_length" warning printed every generate() call
model.generation_config.max_length = None

all_prompts = test_df['prompt'].tolist()
all_ids = test_df['id'].tolist()
BS = CONFIG.get('inference_batch_size', 8)

rows = []
stats = Counter()
t0 = time.time()

for i in range(0, len(all_prompts), BS):
    batch_prompts = all_prompts[i:i + BS]
    batch_ids = all_ids[i:i + BS]
    batch_results = generate_svg_batch(batch_prompts)

    for pid, (svg, status) in zip(batch_ids, batch_results):
        stats[f'status:{status}'] += 1
        if status != 'ok':
            stats['fallback'] += 1
        else:
            stats['valid'] += 1
        stats['total_len'] += len(svg)
        rows.append({'id': pid, 'svg': svg})

    done = min(i + BS, len(all_prompts))
    if done % 50 < BS or done >= len(all_prompts):
        elapsed = time.time() - t0
        rate = done / max(elapsed, 1)
        eta = (len(all_prompts) - done) / max(rate, 0.01)
        print(f'  [{done}/{len(all_prompts)}] {elapsed:.0f}s '
              f'({rate:.2f} it/s, ETA {eta/60:.0f}min)  '
              f'valid={stats["valid"]}  fallback={stats["fallback"]}')

sub_df = pd.DataFrame(rows)
sub_df.to_csv(CONFIG['submission_path'], index=False)

elapsed_min = (time.time() - t0) / 60
n = len(sub_df)
print(f'\nSubmission saved: {CONFIG["submission_path"]}')
print(f'Total rows:    {n}')
print(f'Valid:         {stats["valid"]} ({100*stats["valid"]/max(n,1):.1f}%)')
print(f'Fallback:      {stats["fallback"]} ({100*stats["fallback"]/max(n,1):.1f}%)')
print(f'Avg SVG len:   {stats["total_len"]/max(n,1):.0f} chars')
print(f'Runtime:       {elapsed_min:.1f} min')

print(f'\n--- Failure distribution ---')
for k, v in sorted(stats.items()):
    if k.startswith('status:'):
        print(f'  {k}: {v}')

sub_df.head()

Test prompts: 1000
  [52/1000] 1067s (0.05 it/s, ETA 324min)  valid=52  fallback=0
  [100/1000] 2075s (0.05 it/s, ETA 311min)  valid=100  fallback=0
  [152/1000] 3104s (0.05 it/s, ETA 289min)  valid=152  fallback=0
  [200/1000] 4084s (0.05 it/s, ETA 272min)  valid=200  fallback=0
  [252/1000] 5266s (0.05 it/s, ETA 261min)  valid=251  fallback=1
  [300/1000] 6227s (0.05 it/s, ETA 242min)  valid=299  fallback=1
  [352/1000] 7307s (0.05 it/s, ETA 224min)  valid=351  fallback=1
  [400/1000] 8289s (0.05 it/s, ETA 207min)  valid=398  fallback=2
  [452/1000] 9395s (0.05 it/s, ETA 190min)  valid=448  fallback=4
  [500/1000] 10378s (0.05 it/s, ETA 173min)  valid=496  fallback=4
  [552/1000] 11460s (0.05 it/s, ETA 155min)  valid=548  fallback=4
  [600/1000] 12514s (0.05 it/s, ETA 139min)  valid=595  fallback=5
  [652/1000] 13661s (0.05 it/s, ETA 122min)  valid=647  fallback=5
  [700/1000] 14716s (0.05 it/s, ETA 105min)  valid=694  fallback=6
  [752/1000] 15906s (0.05 it/s, ETA 87min)  valid=746 

,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" fill=""..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" fill=""..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" fill=""..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" fill=""..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" fill=""..."


---
## Summary & Notes

**What this baseline does:**

- Loads competition `train.csv` (no external datasets), normalizes SVGs to 256×256, filters by quality (≤2 500 chars)
- Fine-tunes Qwen2.5-3B-Instruct with QLoRA (`r=16`, 4-bit, `packing=False`, `response_template` for loss masking)
- Batched inference (`inference_batch_size=4`) with conservative decoding (`temperature=0.3`, `top_p=0.8`)
- Full post-processing chain: extract → sanitize → validate → fallback
- Tracks per-layer failure reasons (`extract_fail`, `sanitize_fail`, `validate_fail:<reason>`)

**Two-phase design:**

- **Phase A (Training):** Cells 1-15. Requires internet to load model weights. Saves LoRA adapter to `output_dir`.
- **Phase B (Inference):** Cells 16-21. Reloads model without Unsloth patches for clean generation.

**Key parameters to tune next:**

| Parameter | Current | Try |
|---|---|---|
| `max_train_samples` | 12 000 | 20 000-29 000 |
| `max_svg_train_chars` | 2 500 | 3 000-4 000 (with monitoring) |
| `lora_r` | 16 | 32, 64 |
| `num_train_epochs` | 1 | 2-3 |
| `temperature` | 0.3 | 0.5-0.7 (after valid rate ≥ 80%) |
| `inference_batch_size` | 4 | 8 (if VRAM allows) |